# Extending MACE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/extending_mace.ipynb)


**Lennard-Jones Centre · University of Cambridge**

A hands-on tutorial for first-year PhD students in computational chemistry. Follow an extension from its scientific motivation and symmetry requirements through architecture, code, training data, inference and validation.

## Guiding question

When ordinary short-range MACE does not express the physics or property we need, which parts of the architecture and software path should change?

## The four examples

1. **MACEField:** condition energy on an applied electric field and derive polarization response.
2. **AtomicDipolesMACE:** add a vector-property readout.
3. **Magnetic MACE:** make energy depend on atomic moment vectors.
4. **MACELES:** add a long-range electrostatic energy branch.

**Time:** about 2–3 hours if you run the optional pretraining and checkpoint demonstrations. Each architecture figure adapts the original MACE theory schematic; each example follows the change through theory, source code, training and a structural probe.

> **Teaching data:** the BTO section uses copied BaTiO₃ reference-calculation labels. The water and magnetic extension labs use deterministic analytic examples. The BTO data have narrow field coverage and are not an experimental or validated hysteresis dataset.


## Background and recommended reading

This tutorial builds on the Cambridge MLIP notebooks and the MACE training and theory tutorials. It assumes familiarity with energy-and-force fitting, equivariance, irreducible representations, message passing and product bases.

### Cambridge 07-MLIPs

The [Cambridge 07-MLIPs sequence](https://gitlab.com/cam-ml/tutorials/-/tree/main/notebooks/07-mlips?ref_type=heads) provides practical MLIP background:

- [MACE in Practice I](https://gitlab.com/cam-ml/tutorials/-/blob/main/notebooks/07-mlips/T01_MACE_Practice_I.ipynb)
- [MACE in Practice II](https://gitlab.com/cam-ml/tutorials/-/blob/main/notebooks/07-mlips/T02_MACE_Practice_II.ipynb)
- [MLIP applications](https://gitlab.com/cam-ml/tutorials/-/blob/main/notebooks/07-mlips/T03_MLIP_Apps.ipynb)
- [MACE NEB practice](https://gitlab.com/cam-ml/tutorials/-/blob/main/notebooks/07-mlips/T04_MACE_neb.ipynb)

### MACE training and theory

Use the [MACE tutorial index](https://mace-docs.readthedocs.io/en/latest/examples/tutorials.html) for training and evaluation. For the architecture derivation, read [Tutorial 3: MACE Theory](https://github.com/ACEsuit/mace-tutorials/blob/main/T03_MACE_Theory.ipynb), which develops the equations alongside the implementation.

The base architecture image in this notebook is adapted from Tutorial 3. Its source is MIT-licensed; figure attribution is recorded in `tutorials/figures/ATTRIBUTION.md`.

## Learning objectives

We begin with MACEField, then compare direct dipole, magnetic and long-range extensions. By the end of the notebook, you should be able to:

1. Choose whether a new physical ingredient belongs in model inputs, equivariant message passing, a readout, energy assembly or the calculator.
2. Trace a model through the parser, factory, forward outputs, loss, checkpoint and inference path.
3. Design a structural probe that varies one physical quantity at a time, inspect the structures and plot the response.
4. Check symmetry, units, derivatives, data coverage and matched controls before interpreting a result.

**Suggested route:** read the theory and source sections, then run the geometry cells. The bundled ExtXYZ data and training scripts can produce small checkpoints. Set the checkpoint path in the configuration cell before running each inference demonstration. Predictions from random weights are omitted because they are not scientifically interpretable.

## The shared starting point: MACE Theory Tutorial 3

The original schematic from MACE Theory Tutorial 3 is displayed after setup.

The architecture has three repeating stages:

1. **Embedding:** chemical species and interatomic geometry become node and edge features.
2. **Interaction:** equivariant operations collect information from neighbouring atoms.
3. **Product and update:** features combine to represent higher-body correlations and are passed to the next layer.

A readout maps the final representation to the model's target. The four figures below keep this shared scaffold and add the extension-specific inputs, modules or output branches.

## Set up the notebook and teaching data

Open this notebook in Colab with the badge above, or run it from a MACE-Field checkout. In Colab the first code cell clones the MACEField `clean` branch and this tutorial repository and installs the editable package plus visualization dependencies. The notebook and source are loaded from GitHub; no Drive mount or manually prepared ZIP is required.

Checkpoint files are intentionally not stored in Git. In Colab, each checkpoint is trained when its inference section is first reached if it is missing. Training defaults to 100 epochs; set `MACE_TUTORIAL_EPOCHS=10` before running a section for a short code-path demonstration. Short fits are not accuracy results. Checkpoints and outputs stay in the temporary Colab runtime unless you download them.

The BTO section uses the 59-frame reference-calculation dataset and fixed 48/11 split. Its local provenance does not retain the electronic-structure code or functional. Other bundled extension data are small analytic teaching examples.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def clone_checkout(path, url, branch):
    path = Path(path)
    if not (path / ".git").is_dir():
        if path.exists() and any(path.iterdir()):
            raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", branch, url, str(path)],
            check=True,
        )


if IN_COLAB:
    REPO = Path("/content/mace-field")
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    clone_checkout(REPO, PACKAGE_URL, "clean")
    clone_checkout(TUTORIAL_REPO, TUTORIAL_URL, "main")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--editable", str(REPO)],
        check=True,
    )
else:
    CWD = Path.cwd().resolve()
    candidates = (CWD, *CWD.parents)
    TUTORIAL_REPO = next(
        (path for path in candidates
         if (path / "tutorials/scripts/bootstrap.py").is_file()),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run from a mace-field-tutorial checkout, or open this notebook in Colab."
        )
    package_candidates = [
        Path(os.environ["MACEFIELD_REPO"]).expanduser().resolve()
        if os.environ.get("MACEFIELD_REPO") else None,
        TUTORIAL_REPO.parent / "mace-field",
        *candidates,
    ]
    REPO = next(
        (path.resolve() for path in package_candidates
         if path is not None and (path / "mace/modules/extensions.py").is_file()),
        None,
    )
    if REPO is None:
        raise RuntimeError(
            "Clone mdi-group/mace-field beside this tutorial checkout, or set MACEFIELD_REPO."
        )
    os.environ["MACEFIELD_REPO"] = str(REPO)

os.environ["MACEFIELD_REPO"] = str(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(TUTORIAL_REPO))
print("MACEField source:", REPO)
print("Tutorial assets:", TUTORIAL_REPO)
print("Runtime:", "Google Colab" if IN_COLAB else "local Jupyter")

from tutorials.scripts.bootstrap import ensure
ensure("base,visualization")

from itertools import product
import inspect
import numpy as np
import matplotlib.pyplot as plt
import torch
from ase import Atoms
from ase.data import covalent_radii
from ase.io import read
from ase.neighborlist import neighbor_list

from mace.calculators import MACECalculator, MagneticMACECalculator
from mace.modules.models import AtomicDipolesMACE
from mace.modules.extensions import MACEField, MACELES, MagneticScaleShiftMACE

import py3Dmol
import ipywidgets as widgets

PY3DMOL_AVAILABLE = True
WIDGETS_AVAILABLE = True
print("Python:", sys.version.split()[0])
print("Interactive structure viewer and widgets are ready.")

display(Image(filename=str(TUTORIAL_REPO / "tutorials/figures/mace_theory_architecture.png"), width=760))


In [ ]:
from html import escape
from IPython.display import HTML, Image, display


def show_source_hits(obj, needles, context=5, max_chars=12000):
    """Show relevant implementation windows in a collapsible source excerpt."""
    lines, first_line = inspect.getsourcelines(obj)
    hit_rows = [i for i, line in enumerate(lines) if any(needle in line for needle in needles)]
    spans = []
    for row in hit_rows:
        lo, hi = max(0, row - context), min(len(lines), row + context + 1)
        if spans and lo <= spans[-1][1]:
            spans[-1] = (spans[-1][0], max(spans[-1][1], hi))
        else:
            spans.append((lo, hi))
    output = []
    for lo, hi in spans:
        output.extend(f"{first_line + i:5d}  {lines[i]}" for i in range(lo, hi))
        output.append("       ...")
    text = "".join(output)
    limited = len(text) > max_chars
    text = text[:max_chars]
    note = "Excerpt limited; inspect the full method in the named file." if limited else "Source from the current checkout."
    display(HTML(
        '<details>'
        f'<summary>Open {escape(obj.__qualname__)} source windows</summary>'
        f'<p>{note}</p>'
        f'<pre><code>{escape(text)}</code></pre></details>'
    ))


def read_source(relative_path, start, stop):
    """Show a numbered source range from this checkout in a collapsible excerpt."""
    path = REPO / relative_path
    lines = path.read_text().splitlines()
    text = "\n".join(f"{number:5d}  {lines[number - 1]}" for number in range(start, min(stop, len(lines)) + 1))
    display(HTML(
        '<details>'
        f'<summary>Open <code>{escape(str(path.relative_to(TUTORIAL_REPO)))}</code>, lines {start}–{stop}</summary>'
        f'<pre><code>{escape(text)}</code></pre></details>'
    ))


In [ ]:
MODEL_PATHS = {
    "field": TUTORIAL_REPO / "tutorials/macefield/runs/bto_macefield_training",
    "dipole": TUTORIAL_REPO / "tutorials/models/atomicdipoles",
    "magnetic": TUTORIAL_REPO / "tutorials/models/magnetic",
    "les": TUTORIAL_REPO / "tutorials/models/maceles",
    "short_range": TUTORIAL_REPO / "tutorials/models/maceles_short_range",
}
TRAINING_SCRIPTS = {
    "field": TUTORIAL_REPO / "tutorials/macefield/scripts/train_macefield.sh",
    "dipole": TUTORIAL_REPO / "tutorials/scripts/train_atomicdipoles.sh",
    "magnetic": TUTORIAL_REPO / "tutorials/scripts/train_magnetic.sh",
    "les": TUTORIAL_REPO / "tutorials/scripts/train_maceles.sh",
    "short_range": TUTORIAL_REPO / "tutorials/scripts/train_maceles_short_range.sh",
}
MODEL_DTYPE = "float64"
import torch
DEVICE = os.environ.get(
    "MACE_TUTORIAL_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"
)
AUTO_TRAIN = os.environ.get("MACE_TUTORIAL_AUTO_TRAIN", "1" if IN_COLAB else "0") == "1"


def _latest_checkpoint(path):
    path = Path(path)
    if path.is_file():
        return path
    if path.is_dir():
        checkpoints = sorted(path.glob("*.model"), key=lambda p: p.stat().st_mtime, reverse=True)
        if checkpoints:
            return checkpoints[0]
    return None


def require_model(name):
    if name not in MODEL_PATHS:
        raise KeyError(f"Unknown tutorial checkpoint: {name}")
    output_dir = Path(MODEL_PATHS[name])
    checkpoint = _latest_checkpoint(output_dir)
    if checkpoint is not None:
        return checkpoint

    if AUTO_TRAIN:
        script = Path(TRAINING_SCRIPTS[name])
        env = os.environ.copy()
        env.update({
            "PYTHON": sys.executable,
            "DEVICE": DEVICE,
            "EPOCHS": os.environ.get("MACE_TUTORIAL_EPOCHS", "100"),
            "OUT_DIR": str(output_dir),
        })
        print(f"Training the missing {name} teaching checkpoint ({env['EPOCHS']} epochs) at {output_dir}")
        subprocess.run(["bash", str(script)], cwd=TUTORIAL_REPO, env=env, check=True)
        checkpoint = _latest_checkpoint(output_dir)
        if checkpoint is not None:
            return checkpoint

    command = f"OUT_DIR={output_dir} EPOCHS=100 DEVICE={DEVICE} bash {TRAINING_SCRIPTS[name]}"
    raise FileNotFoundError(
        f"No {name} checkpoint in {output_dir}. Set MACE_TUTORIAL_AUTO_TRAIN=1 or run: {command}"
    )

print("Compute device:", DEVICE, "| automatic training for missing checkpoints:", AUTO_TRAIN)
print("Model locations:", {name: str(path.relative_to(TUTORIAL_REPO)) for name, path in MODEL_PATHS.items()})


#### A MACE extension spans the software stack

A new `torch.nn.Module` is only one part of a trainable, loadable MACE extension. The path is:

**scientific target → parser and factory → model and outputs → loss and checkpoint → calculator and validation**

The parser must accept the model name, the factory must construct the right class and blocks, training must request compatible outputs and a loss, and inference must route inputs to the correct calculator. The dipole path has a naming exception: training selects `AtomicDipolesMACE`, while the ASE calculator uses `model_type="DipoleMACE"`. Magnetic MACE uses `MagneticMACECalculator`; MACEField and MACELES use the general energy calculator.

The expandable source excerpts below show the dispatch in this checkout.

In [ ]:
read_source("mace/tools/arg_parser.py", 145, 162)
read_source("mace/tools/model_script_utils.py", 279, 316)
read_source("mace/tools/model_script_utils.py", 408, 505)

In [ ]:
read_source("mace/tools/scripts_utils.py", 870, 940)
read_source("mace/calculators/mace.py", 220, 258)
read_source("mace/calculators/mace.py", 1117, 1210)

> **Trace the software route:** for each model, identify the parser choice, factory branch, model class, objective and output selection, and calculator. Mark each component reused from base MACE. Then choose one additional input and trace where it must be declared, read, batched and passed to `forward`. A feature that works inside the module but not through the calculator is not yet a usable extension.

#### Explore molecules and crystals in 3D

The interactive viewer uses 3Dmol to show ball-and-stick structures, periodic cells, per-atom values and vector arrows. The dropdown starts with the BaTiO₃ field-response data, then lets you browse the other bundled training sets.

If the viewer or widgets are missing, the setup cell installs them automatically. You can also run `python -m tutorials.scripts.bootstrap --feature visualization` or run `python -m tutorials.scripts.bootstrap --feature visualization` from the tutorial checkout.

In [ ]:
ELEMENT_COLORS = {
    "H": "#d8dee8", "C": "#364152", "N": "#2878b5", "O": "#d94f45",
    "Fe": "#ce762b", "Ba": "#4d9a64", "Ti": "#8555a6",
}

def _xyz_record(atoms, title):
    rows = [str(len(atoms)), title]
    rows.extend(
        f"{symbol:2s} {x: .8f} {y: .8f} {z: .8f}"
        for symbol, (x, y, z) in zip(atoms.get_chemical_symbols(), atoms.positions)
    )
    return "\n".join(rows) + "\n"

def _point(xyz):
    x, y, z = np.asarray(xyz, dtype=float)
    return {"x": float(x), "y": float(y), "z": float(z)}

def show_structure(
    atoms,
    values=None,
    value_name="value",
    value_units="",
    vectors=None,
    title="Structure",
    width=760,
    height=560,
    arrow_length=1.25,
):
    if not PY3DMOL_AVAILABLE:
        raise ImportError("run python -m tutorials.scripts.bootstrap --feature visualization from the tutorial checkout")
    display(HTML(f"<h4>{escape(title)}</h4>"))
    view = py3Dmol.view(width=width, height=height)
    view.addModel(_xyz_record(atoms, title), "xyz")
    view.setBackgroundColor("#ffffff")
    view.setStyle({}, {"sphere": {"scale": 0.52}})
    for i, symbol in enumerate(atoms.get_chemical_symbols()):
        view.addStyle({"index": i}, {"sphere": {
            "color": ELEMENT_COLORS.get(symbol, "#6688aa"),
            "scale": 0.56 if symbol not in ("H", "O") else 0.48,
        }})

    # Draw chemically plausible nearest-neighbour bonds, including periodic images.
    cutoffs = 1.12 * covalent_radii[atoms.numbers]
    ii, jj, shifts = neighbor_list("ijS", atoms, cutoffs)
    seen = set()
    cell = np.asarray(atoms.cell, dtype=float)
    for i, j, shift in zip(ii, jj, shifts):
        if i >= j:
            continue
        key = (int(i), int(j), tuple(int(s) for s in shift))
        if key in seen:
            continue
        seen.add(key)
        start = atoms.positions[i]
        end = atoms.positions[j] + np.asarray(shift) @ cell
        view.addCylinder({
            "start": _point(start), "end": _point(end), "radius": 0.09,
            "color": "#8291a1", "fromCap": 1, "toCap": 1,
        })

    if values is not None:
        values = np.asarray(values, dtype=float).reshape(-1)
        if len(values) != len(atoms):
            raise ValueError("values must contain one scalar per atom")
        span = max(float(np.max(np.abs(values))), 1.0e-12)
        from matplotlib.colors import to_hex
        cmap = plt.get_cmap("coolwarm")
        for i, value in enumerate(values):
            color = to_hex(cmap(0.5 + 0.5 * float(value) / span))
            view.addStyle({"index": i}, {"sphere": {"color": color, "scale": 0.68}})
            units = f" {value_units}" if value_units else ""
            view.addLabel(
                f"{value_name}={value:+.2f}{units}",
                {
                    "position": _point(atoms.positions[i] + np.array([0.0, 0.0, 0.32])),
                    "fontColor": "#111827",
                    "backgroundColor": "#ffffff",
                    "borderColor": color,
                    "borderThickness": 1,
                    "fontSize": 13,
                    "showBackground": True,
                },
            )
        units_text = escape(value_units) if value_units else "model units"
        display(HTML(
            f"<p>Atom color and label show {escape(value_name)} in {units_text}; "
            "the scale is symmetric about zero.</p>"
        ))
    else:
        for i, symbol in enumerate(atoms.get_chemical_symbols()):
            view.addStyle({"index": i}, {
                "sphere": {
                    "color": ELEMENT_COLORS.get(symbol, "#6688aa"),
                    "scale": 0.56 if symbol not in ("H", "O") else 0.48,
                }
            })

    if np.any(atoms.pbc):
        corners = {
            bits: np.asarray(bits, dtype=float) @ cell
            for bits in product((0, 1), repeat=3)
        }
        for bits, start in corners.items():
            for axis in range(3):
                if bits[axis] == 0:
                    other = bits[:axis] + (1,) + bits[axis + 1:]
                    view.addLine({
                        "start": _point(start), "end": _point(corners[other]),
                        "color": "#536273", "linewidth": 2.0,
                    })

    if vectors:
        for entry in vectors:
            origin, vector, label, color = entry[:4]
            units = entry[4] if len(entry) > 4 else ""
            start = np.asarray(origin, dtype=float)
            vector = np.asarray(vector, dtype=float)
            magnitude = float(np.linalg.norm(vector))
            if magnitude > 1.0e-14:
                end = start + arrow_length * vector / magnitude
                view.addArrow({
                    "start": _point(start), "end": _point(end), "color": color,
                    "radius": 0.13, "radiusRatio": 2.5,
                })
                display_label = f"{label}: |v|={magnitude:.3g} {units}".strip()
                view.addLabel(
                    display_label,
                    {
                        "position": _point(end + np.array([0.0, 0.0, 0.18])),
                        "fontColor": color,
                        "backgroundColor": "#ffffff",
                        "fontSize": 14,
                        "showBackground": True,
                    },
                )
        display(HTML(
            "<p>Arrows show direction with a fixed display length; labels give "
            "the vector magnitude in its model units.</p>"
        ))

    view.zoomTo()
    return view

def show_reference_data(path, n=3):
    frames = read(path, index=f":{n}")
    for i, atoms in enumerate(frames):
        print(
            i, atoms.get_chemical_formula(), "n=", len(atoms), "pbc=", atoms.pbc,
            "info=", sorted(atoms.info),
            "arrays=", sorted(set(atoms.arrays) - {"numbers", "positions"}),
        )

In [ ]:
if WIDGETS_AVAILABLE and PY3DMOL_AVAILABLE:
    _data_choices = {
        "BaTiO3 · MACEField reference data": "macefield/data/macefield_batio3_reference_train.extxyz",
        "Water dipoles · molecule": "atomicdipoles_water_train.extxyz",
        "Fe dimer · magnetic moments": "magnetic_fe2_train.extxyz",
        "Water dimer · LES": "maceles_water_dimer_train.extxyz",
    }

    def _preview_training_set(choice):
        dataset = TUTORIAL_REPO / "tutorials" / _data_choices[choice]
        frames = read(dataset, index=":")
        atoms = frames[0]
        print(f"{dataset.name} · {len(atoms)} atoms · {len(frames)} frames")
        display(show_structure(atoms, title=choice))

    data_picker = widgets.Dropdown(options=list(_data_choices), description="Explore:", layout=widgets.Layout(width="420px"))
    display(widgets.interactive(_preview_training_set, choice=data_picker))
else:
    print("For the interactive data picker, run python -m tutorials.scripts.bootstrap --feature visualization from the tutorial checkout")

#### Keep the demonstrations physically honest

> A checkpoint only makes sense with the species, units, data fields, cutoff, boundary conditions and model head for which it was trained. The example molecules and crystals below are probes, not reference quantum-chemistry structures. A smooth curve from one checkpoint is not by itself validation; compare against labels, symmetry constraints, derivatives and a matched control.

## Extension 1 — MACEField: make the energy depend on an electric field

### 1.1 What changes in the architecture?

MACEField adds an external polar vector to the equivariant message-passing
calculation. The shared MACE trunk still processes atomic environments; field
couplings enter intermediate hidden features, and one field-dependent scalar
energy remains the source of forces and electrical response.

The architecture diagram below highlights where the external field enters.

The key design decision is that the field is an input to the energy model.
Polarization and higher response properties are derivatives of that same
energy. This gives a consistent route for forces, field response and mixed
derivatives.

In [ ]:
display(Image(filename=str(TUTORIAL_REPO / "tutorials/figures/architecture_macefield.png"), width=760))

### 1.2 Theory, symmetries and units

For a periodic cell of volume Ω, this implementation defines polarization
density as

$$\mathbf P=-\Omega^{-1}\frac{\partial E}{\partial\boldsymbol{\mathcal E}}.$$

The Born effective charge tensor is
$Z^*_{i,\alpha\beta}=\Omega\,\partial P_\alpha/\partial u_{i\beta}$ in units
of $e$. The returned tensor named polarizability is normalized by vacuum
permittivity and cell volume: it is the dimensionless electric susceptibility
$\chi_e=\epsilon_0^{-1}\partial\mathbf P/\partial\mathbf E$. The relative
dielectric tensor is $\epsilon_r=I+\chi_e$. It is not the molecular
polarizability in Å³.

Rotate the structure and field together: energy is invariant, $\mathbf P$
rotates as a vector, and $Z^*$ and $\chi_e$ transform as rank-two tensors.
For a periodic crystal, keep the cell and polarization branch fixed. The
Born-charge acoustic sum rule is $\sum_i Z_i^*=0$.

### 1.3 Follow the implementation

The field has shape one vector per graph, is broadcast to atoms, and is coupled
to hidden irreducible representations with tensor products. The final scalar
energy is differentiated to obtain polarization; second derivatives produce
Born charges and electric susceptibility. Read the live source windows and
trace how the field argument reaches those operations.

In [ ]:
show_source_hits(
    MACEField.__init__,
    needles=("field_feats", "field_linear", "num_interactions - 1"),
    context=6,
)
show_source_hits(
    MACEField.forward,
    needles=("electric_field", "per_atom_electric_field", "field_feat(", "get_polarization(", "get_becs", "get_polarizability"),
    context=5,
    max_chars=12000,
)

#### Questions to answer from the source

1. Why is the field a graph-level vector before it is broadcast to atoms?
2. Which tensor-product paths preserve rotation covariance?
3. Why are field couplings added between interaction/product stages?
4. Which derivatives correspond to $P$, $Z^*$, and $\chi_e$?
5. How does inference handle response requests when a training batch has no
   active response labels?

### 1.4 BaTiO₃ reference data for MACEField

The MACEField example uses 59 periodic 135-atom structures copied from `MACEField_dev/BaTiO3/BaTiO3-preprocessed.xyz`. The seed-23 split contains 48 training and 11 validation frames. The local preparation notebook transfers stored ASE calculator results to energy, force, stress, polarization, Born-charge, susceptibility and field labels; it does not preserve the electronic-structure code or functional.

All frames have the same field tag, `[0.0036, 0.0036, 0.0036] V/Å`. This is real calculated reference data, but it has no field reversal or switching-path coverage. The validation split checks held-out structures at the same field and cannot establish a hysteresis prediction. See `tutorials/macefield/data/macefield_batio3_reference_split.json` for checksums and split provenance.

In [ ]:
import json
field_train = read(TUTORIAL_REPO / "tutorials/macefield/data/macefield_batio3_reference_train.extxyz", index=":")
field_valid = read(TUTORIAL_REPO / "tutorials/macefield/data/macefield_batio3_reference_valid.extxyz", index=":")
field_manifest = json.loads((TUTORIAL_REPO / "tutorials/macefield/data/macefield_batio3_reference_split.json").read_text())
print("train / validation frames:", len(field_train), "/", len(field_valid))
print("source SHA-256:", field_manifest["source_sha256"])
print("field vectors (V/Å):", np.unique(np.round([
    a.info["REF_electric_field"] for a in field_train
], 10), axis=0))
print("available labels:", sorted(field_train[0].info), sorted(field_train[0].arrays))

### 1.5 Inspect a reference structure

The reference structures are 3×3×3 perovskite supercells with 135 atoms. Inspect one actual labelled configuration instead of replacing the data with a hand-built primitive-cell fixture.

In [ ]:
bto_reference = field_train[0].copy()
print(bto_reference.get_chemical_formula(), "atoms:", len(bto_reference),
      "cell volume (Å³):", bto_reference.get_volume())
print("field (V/Å):", bto_reference.info["REF_electric_field"])
print("reference polarization (e/Å²):", bto_reference.info["REF_polarization"])
show_structure(bto_reference, title="BaTiO₃ reference-calculation structure")

### 1.6 Check the energy derivative near the labelled field

Once notebook 02 has trained a model, compare autograd polarization with a finite-difference slope of the same model energy. Perturb the labelled field along its `[111]` direction. This checks the calculator's sign and volume normalization; it does not test field transfer or physical accuracy.

In [ ]:
field_calc_path = require_model("field")
field0 = np.asarray(bto_reference.info["REF_electric_field"], dtype=float)
field_direction = field0 / np.linalg.norm(field0)
field_offsets = np.linspace(-2.0e-4, 2.0e-4, 9)
field_energies, p_parallel = [], []
calc = MACECalculator(
    model_paths=str(field_calc_path), model_type="MACEField",
    device=DEVICE, default_dtype=MODEL_DTYPE,
    electric_field=field0,
    compute_polarization=True, compute_becs=False, compute_polarizability=False,
)
for offset in field_offsets:
    calc.electric_field = field0 + offset * field_direction
    calc.reset()
    bto_reference.calc = calc
    field_energies.append(bto_reference.get_potential_energy())
    p_parallel.append(float(np.dot(calc.results["polarization"], field_direction)))
field_energies = np.asarray(field_energies)
p_parallel = np.asarray(p_parallel)
p_fd = -np.gradient(field_energies, field_offsets) / bto_reference.get_volume()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(field_offsets, field_energies - field_energies.min(), "o-", color="#315a7d")
axes[0].set(xlabel="Field perturbation along [111] (V/Å)", ylabel="Relative energy (eV)")
axes[1].plot(field_offsets, p_parallel, "o-", label="Autograd polarization")
axes[1].plot(field_offsets, p_fd, "s--", label="− finite-difference dE/dE / Ω")
axes[1].set(xlabel="Field perturbation (V/Å)", ylabel="Polarization projection (e Å⁻²)")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
fig.suptitle("MACEField derivative check on a reference structure")
fig.tight_layout()
plt.show()
print("max |P − finite-difference P| (e Å⁻²):",
      float(np.max(np.abs(p_parallel - p_fd))))

### 1.7 What the field labels do not support

The reference set contains one positive field vector, not a path through positive and negative fields. There is no reference basis for tracing metastable minima, identifying a switching field, or validating a polarization loop. The field-cycle workflow in notebook 03 is therefore presented only as an exploratory software exercise.

In [ ]:
field_vectors = np.asarray([a.info["REF_electric_field"] for a in field_train], dtype=float)
unique_fields = np.unique(np.round(field_vectors, decimals=10), axis=0)
print("unique labelled field vectors:", len(unique_fields))
print(unique_fields)
print("Independent test split:", field_manifest["test_frames"])
print("A physical switching field cannot be estimated from this coverage.")

In [ ]:
field_direction = field0 / np.linalg.norm(field0)
train_energy_pa = np.asarray([a.info["REF_energy"] / len(a) for a in field_train])
valid_energy_pa = np.asarray([a.info["REF_energy"] / len(a) for a in field_valid])
train_p_parallel = np.asarray([np.dot(a.info["REF_polarization"], field_direction) for a in field_train])
valid_p_parallel = np.asarray([np.dot(a.info["REF_polarization"], field_direction) for a in field_valid])
energy_offset = np.mean(train_energy_pa)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter((train_energy_pa - energy_offset) * 1000, train_p_parallel, s=26, label="train")
ax.scatter((valid_energy_pa - energy_offset) * 1000, valid_p_parallel, s=36, marker="s", label="validation")
ax.set(xlabel="Centered energy (meV/atom)", ylabel="P along labelled field (e/Å²)",
       title="Structural variation at one field vector")
ax.grid(alpha=0.22)
ax.legend()
plt.show()

The plot shows variation across held-out structures, not a field sweep. Since the field coordinate is constant, a low validation error here cannot establish a correct response to field reversal.

Use the reported validation errors as a fit diagnostic for this split. Do not use a zero-field displacement scan as a target: the copied data do not contain a switching-path reference surface.

### 1.8 Born effective charges and dielectric susceptibility

The response arrays have per-atom BEC tensors and a cell susceptibility tensor. Compare model derivatives with the stored reference labels on a held-out training structure. Notebook 02 supervises these higher-order labels only when `MACEFIELD_TUTORIAL_FULL_RESPONSES=1`; otherwise this is an unsupervised response diagnostic.

In [ ]:
response_atoms = field_train[0].copy()
reference_field = np.asarray(response_atoms.info["REF_electric_field"], dtype=float)
response_atoms.calc = MACECalculator(
    model_paths=str(field_calc_path), model_type="MACEField",
    device=DEVICE, default_dtype=MODEL_DTYPE,
    electric_field=reference_field,
    compute_polarization=True, compute_becs=True, compute_polarizability=True,
)
response_atoms.get_potential_energy()
becs = np.asarray(response_atoms.calc.results["becs"]).reshape(len(response_atoms), 3, 3)
chi_e = np.asarray(response_atoms.calc.results["polarizability"]).reshape(3, 3)
reference_becs = np.asarray(response_atoms.arrays["REF_becs"]).reshape(len(response_atoms), 3, 3)
chi_ref = np.asarray(response_atoms.info["REF_polarizability"]).reshape(3, 3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(reference_becs[:, 2, 2], becs[:, 2, 2], s=18)
lo = min(reference_becs[:, 2, 2].min(), becs[:, 2, 2].min())
hi = max(reference_becs[:, 2, 2].max(), becs[:, 2, 2].max())
axes[0].plot([lo, hi], [lo, hi], "k--", lw=1)
axes[0].set(xlabel="Reference Z*zz (e)", ylabel="Model Z*zz (e)", title="Born charges")
image = axes[1].imshow(0.5 * (chi_e + chi_e.T), cmap="viridis")
axes[1].set(xticks=range(3), yticks=range(3), xticklabels=["x", "y", "z"],
            yticklabels=["x", "y", "z"], xlabel="Field", ylabel="Polarization",
            title="Model susceptibility χe")
fig.colorbar(image, ax=axes[1], label="dimensionless")
axes[2].scatter(chi_ref.ravel(), chi_e.ravel(), s=40)
axes[2].set(xlabel="Reference χe", ylabel="Model χe", title="Susceptibility components")
for ax in axes:
    ax.grid(alpha=0.22)
fig.tight_layout()
plt.show()
asr = float(np.linalg.norm(becs.sum(axis=0)))
print("BEC component RMSE (e):", float(np.sqrt(np.mean((becs - reference_becs)**2))))
print("susceptibility component RMSE:", float(np.sqrt(np.mean((chi_e - chi_ref)**2))))
print("model BEC acoustic sum-rule residual (e):", asr)
print("susceptibility symmetry residual:", float(np.linalg.norm(chi_e - chi_e.T)))

The acoustic sum rule and tensor symmetry are internal consistency checks. They do not show that the response matches the reference data. Because the default training run omits BEC and susceptibility loss, response errors can remain large even when polarization was supervised.

In [ ]:
# Finite-difference check of a Ti Z*zz component at the same field.
ti_index = response_atoms.get_chemical_symbols().index("Ti")
delta = 1.0e-3
p_for_bec_fd = []
for displacement in (-delta, delta):
    atoms = field_train[0].copy()
    atoms.positions[ti_index, 2] += displacement
    atoms.calc = MACECalculator(
        model_paths=str(field_calc_path), model_type="MACEField",
        device=DEVICE, default_dtype=MODEL_DTYPE,
        electric_field=reference_field,
        compute_polarization=True, compute_becs=False, compute_polarizability=False,
    )
    atoms.get_potential_energy()
    p_for_bec_fd.append(float(atoms.calc.results["polarization"][2]))
bec_ti_fd = response_atoms.get_volume() * (p_for_bec_fd[1] - p_for_bec_fd[0]) / (2 * delta)
print("Ti Z*zz from autograd:", float(becs[ti_index, 2, 2]))
print("Ti Z*zz from finite differences:", bec_ti_fd)
print("Reference Ti Z*zz:", float(reference_becs[ti_index, 2, 2]))

#### Inspect reference polarization and BEC labels

The structure viewer below uses the stored reference polarization and signed site-resolved (Z^*_{zz}). The displayed configurations come from the actual train split, not from an idealized five-atom motif.

In [ ]:
if WIDGETS_AVAILABLE and PY3DMOL_AVAILABLE:
    @widgets.interact(frame_index=(0, len(field_train) - 1, 1))
    def inspect_reference_state(frame_index=0):
        atoms = field_train[int(frame_index)].copy()
        zstar = np.asarray(atoms.arrays["REF_becs"]).reshape(len(atoms), 3, 3)[:, 2, 2]
        p = np.asarray(atoms.info["REF_polarization"], dtype=float)
        return show_structure(
            atoms,
            values=zstar,
            value_name="Reference Z*zz",
            value_units="e",
            vectors=[(
                atoms.positions.mean(axis=0), p,
                "Reference polarization", "#147d9b", "e Å⁻²",
            )],
            title=f"BaTiO₃ reference frame {frame_index}",
        )
else:
    print("Install the visualization extras to use the structure controls.")

### 1.9 Training contract and extension boundary

The MACEField route needs a model factory, field-aware batching, response-aware forward flags, a loss that consumes active labels, and a calculator that passes the field and exposes derivatives. Notebook 02 trains energy, forces and polarization by default on the copied 48-frame train split with the 11-frame validation split. Set `MACEFIELD_TUTORIAL_FULL_RESPONSES=1` before running notebook 02 to add BEC and susceptibility supervision.

The data still cover only one field vector, so neither training nor derivative consistency validates field reversal or switching. Train from this checkout with:

    bash tutorials/macefield/scripts/train_macefield.sh

The script defaults to CPU and 100 epochs with batch size 1 for the 135-atom cells. Use `DEVICE=cuda`, `EPOCHS=...` or `BATCH_SIZE=...` to adjust those values. The other extension labs keep their bundled analytic examples.

In [ ]:
MACEFIELD_DATA_KEYS = {
    "model": "MACEField",
    "electric_field_key": "REF_electric_field",
    "polarization_key": "REF_polarization",
    "becs_key": "REF_becs",
    "polarizability_key": "REF_polarizability",
    "loss": "UniversalField",
}
MACEFIELD_DATA_KEYS

## Extension 2 — AtomicDipolesMACE: predict a vector property from MACE features

### 2.1 Scientific question and architecture change

> **Architecture choice:** keep the equivariant trunk and add an `l = 1` readout because the target transforms as a vector. This is a direct property readout, not an energy derivative.

Suppose the target is a molecular dipole vector, not an energy. It transforms as a **polar vector**: under a proper rotation Q, the prediction must change from μ to Qμ. A scalar energy readout cannot be reused unchanged as three unrelated Cartesian regressors without breaking this transformation rule.

AtomicDipolesMACE reuses MACE's species embedding, radial and angular geometry, equivariant interactions, and product basis. Its head reads the vector (`l = 1`) part of the equivariant features. In this implementation a readout is attached at successive interaction depths; their atomic contributions are summed. The molecular output is the sum over atoms plus an optional fixed-charge dipole baseline:

$$\boldsymbol{\mu}=\sum_i \boldsymbol{\mu}_i + \sum_i q_i\mathbf{r}_i.$$

This is a **direct property model**. It is not the gradient of a learned energy, so it does not automatically provide conservative forces. The decomposition into atomic contributions is useful for model design and plotting, but individual atomic dipoles are not unique observables.

#### Architecture: add a vector-property readout


The shared species-and-geometry trunk is unchanged. After each interaction–product layer, an equivariant `l = 1` readout predicts an atomic vector contribution. The model sums those contributions over layers, then sums over atoms and adds the fixed-charge baseline to obtain the system dipole. The `node energies` path remains in the inherited MACE schematic for comparison: this `AtomicDipolesMACE` class returns `atomic_dipoles` and `dipole`, not energy or forces.

In [ ]:
display(Image(filename=str(TUTORIAL_REPO / "tutorials/figures/architecture_atomicdipoles.png"), width=760))

### 1.2 Symmetry, units, and boundary conditions

- **Rotation:** rotate a molecule and its reference dipole by the same Q. Compare the full vector, not only its magnitude.
- **Translation:** a neutral molecule's total dipole is origin independent. For a non-neutral object, it is origin dependent; state the chosen origin.
- **Parity:** an electric dipole is polar. Under inversion it changes sign.
- **Charge baseline:** fixed charges contribute Σqᵢrᵢ. Keep charge and coordinate units consistent, and do not interpret learned μᵢ as QM partial charges.
- **Periodic systems:** polarization has a different definition and branch ambiguity in periodic solids. AtomicDipolesMACE is not a replacement for MACEField's periodic polarization path.
- **Calculator interface:** this checkout's ASE interface calls this trained model type <code>DipoleMACE</code>; the training selector is <code>AtomicDipolesMACE</code>. The calculator returns the dipole in the units expected by the checkpoint/training path; verify the dataset convention.

### 1.3 Follow the implementation

The source tells us which irreps the model must retain, which readouts are attached, how atomwise contributions are reduced to a graph-level vector, and which energy-model outputs are explicitly unsupported.

In [ ]:
show_source_hits(
    AtomicDipolesMACE.__init__,
    needles=("LinearDipoleReadoutBlock", "NonLinearDipoleReadoutBlock", "hidden_irreps_out"),
    context=7,
)

show_source_hits(
    AtomicDipolesMACE.forward,
    needles=("readout(node_feats)", "atomic_dipoles", "compute_fixed_charge_dipole", '"dipole"'),
    context=6,
)

#### Trace the forward calculation

As you read the printed source, fill in this chain:

1. <code>node_attrs</code> → node embedding.
2. Positions and periodic shifts → edge vectors, radial basis, and spherical harmonics.
3. Each interaction and product block → updated equivariant node features.
4. Each dipole readout → a per-atom vector contribution at that depth.
5. Contributions are summed over depth and atoms; the fixed-charge baseline is added.
6. The model returns the molecular dipole and atomic dipoles.

Questions for the source: where is the requirement for vector channels enforced? Is the atomwise output invariant or equivariant? Which derivative flags are rejected? Why would it be misleading to call this an energy-and-force model?

### 1.4 Data contract and training route

A direct dipole target must use a clear vector convention and units. The model does not infer the target from forces. The ExtXYZ frames in this tutorial illustrate the required keys; use the exact names declared by your own dataset.

**Pretraining example:** `bash tutorials/scripts/train_atomicdipoles.sh`

- Training data: `tutorials/data/atomicdipoles_water_train.extxyz`
- Validation data: `tutorials/data/atomicdipoles_water_valid.extxyz`
- Defaults: CPU, 100 epochs; set `DEVICE=cuda` or `EPOCHS=...` to change them.
- The script installs only missing extension dependencies.

In [ ]:
DIPOLE_TRAINING_OPTIONS = {
    "model": "AtomicDipolesMACE",
    "loss": "dipole",
    "error_table": "DipoleRMSE",
    "dipole_key": "REF_dipoles",   # choose the key actually stored in your ExtXYZ
    "dipole_weight": 1.0,
    "compute_atomic_dipole": True,
}
DIPOLE_TRAINING_OPTIONS

Before fitting, inspect several frames. Confirm that the target is a length-three vector, its unit is known, the molecular charge and atom order are correct, and train/test splits do not contain conformational duplicates. The value <code>compute_atomic_dipole</code> requests atomic-level output for supported evaluation paths; it does not make atomic dipoles unique or directly observable.

**Dataset exercise:** load your molecular ExtXYZ file with <code>show_reference_data</code>. Choose one water frame and compare its reference dipole with the same structure after a 90° rotation. The rotated target should rotate too.

### 1.5 Build a geometry series and inspect it interactively

The next cells make a water geometry with fixed O–H length and varying H–O–H angle. This isolates one structural coordinate. First view a single geometry without any model.

In [ ]:
def water_at_angle(angle_degrees, bond=0.96, box=12.0):
    theta = np.deg2rad(angle_degrees / 2.0)
    positions = [
        [0.0, 0.0, 0.0],
        [bond * np.cos(theta), bond * np.sin(theta), 0.0],
        [bond * np.cos(theta), -bond * np.sin(theta), 0.0],
    ]
    atoms = Atoms("OH2", positions=positions, cell=[box, box, box], pbc=False)
    atoms.new_array("Qs", np.zeros(len(atoms)))
    return atoms

water_probe = water_at_angle(104.5)
show_structure(water_probe, title="Water probe · fixed O–H bonds, 104.5° angle")

In [ ]:
angles = np.linspace(75.0, 150.0, 40)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(angles, np.full_like(angles, 0.96), color="#4f83a6")
ax.set(xlabel="H–O–H angle (degrees)", ylabel="O–H bond length (Å)",
       title="Probe design: change angle, hold bond lengths fixed")
ax.grid(alpha=0.25)
plt.show()

if WIDGETS_AVAILABLE and PY3DMOL_AVAILABLE:
    @widgets.interact(angle=(75.0, 150.0, 1.0))
    def explore_water_angle(angle=104.5):
        return show_structure(water_at_angle(angle), title=f"Water angle = {angle:.0f}°")

#### Check vector equivariance without a trained checkpoint

This tiny geometry-only cell verifies the rotation rule for a reference vector. It does not test a model; the checkpoint cell below can use the same transformation as a model-level test.

In [ ]:
from scipy.spatial.transform import Rotation

water = water_at_angle(104.5)
Q = Rotation.from_euler("z", 90.0, degrees=True).as_matrix()
water_rotated = water.copy()
water_rotated.positions = water.positions @ Q.T
reference_mu = np.array([0.0, 0.0, 1.0])  # an arbitrary vector for the transformation demonstration
print("vector before:", reference_mu)
print("vector after:", Q @ reference_mu)

### 1.6 Checkpoint demonstration: scan the angle and plot the dipole

Set <code>MODEL_PATHS["dipole"]</code> to a model trained on a compatible species and dataset. The scan uses identical connectivity and bond lengths while opening the angle. Inspect both the components and vector direction; looking only at the magnitude can hide a symmetry error.

In [ ]:
dipole_calc = MACECalculator(
    model_paths=str(require_model("dipole")),
    model_type="DipoleMACE",
    device=DEVICE, default_dtype=MODEL_DTYPE,
)
angle_grid = np.linspace(80.0, 145.0, 18)
dipoles = []
for angle in angle_grid:
    atoms = water_at_angle(angle)
    atoms.calc = dipole_calc
    dipoles.append(np.asarray(dipole_calc.get_property("dipole", atoms)))
dipoles = np.asarray(dipoles)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for component, label in enumerate(("x", "y", "z")):
    axes[0].plot(angle_grid, dipoles[:, component], marker="o", ms=3, label=label)
axes[0].set(xlabel="H–O–H angle (degrees)", ylabel="dipole component (checkpoint units)")
axes[0].legend(title="component")
axes[0].grid(alpha=0.25)
axes[1].plot(angle_grid, np.linalg.norm(dipoles, axis=1), marker="o", ms=3)
axes[1].set(xlabel="H–O–H angle (degrees)", ylabel="|μ| (checkpoint units)")
axes[1].grid(alpha=0.25)
fig.suptitle("AtomicDipolesMACE: fixed O–H length, varied angle")
plt.tight_layout()

In [ ]:
angle_to_view = 104.5
atoms = water_at_angle(angle_to_view)
atoms.calc = dipole_calc
mu = np.asarray(dipole_calc.get_property("dipole", atoms))
show_structure(atoms, vectors=[(atoms.positions.mean(axis=0), mu, "predicted dipole", "#d97706", "e Å")],
               title=f"Predicted water dipole at {angle_to_view:.1f}°")

#### Model-level rotation test

Rotate the geometry and compare μ(QR) with Qμ(R). The numerical error should be judged relative to model precision and prediction scale. For charged examples, rotate about the same defined origin and include the same charges.

In [ ]:
angle = 104.5
atoms = water_at_angle(angle)
mu_original = np.asarray(dipole_calc.get_property("dipole", atoms))
Q = Rotation.from_euler("xyz", [37.0, 15.0, 68.0], degrees=True).as_matrix()
rotated = atoms.copy()
rotated.positions = atoms.positions @ Q.T
mu_rotated = np.asarray(dipole_calc.get_property("dipole", rotated))
print("equivariance residual (model units):", np.linalg.norm(mu_rotated - Q @ mu_original))

#### Diagnose a failure

If the residual is large, check that (a) the atom order was unchanged, (b) the vector was rotated using the same active/passive convention as the coordinates, (c) the charges and origin are consistent, (d) the chosen calculator/model type is correct, and (e) the training targets use one Cartesian convention. Plot μ components on a rotated series rather than comparing only |μ|.

## Extension 3 — Magnetic MACE: let magnetic moments change the energy

### 3.1 Scientific question and architecture change

> **Architecture choice:** magnetic moments enter before the scalar energy readout, so geometry and spin state can jointly change the energy surface.

For a magnetic material, geometry alone may not determine the energy. Atomic moment vectors mᵢ matter: two structures with the same positions can have different energies for different moment magnitudes or alignments. The target is a scalar energy surface `E(R, m)`; forces and magnetic forces are its derivatives.

This is not a new terminal vector head. `MagneticScaleShiftMACE` adds moment-dependent radial and angular features and passes them through magnetic-aware interaction and product blocks. The learned output is still a scalar energy, now conditional on moment inputs. A fixed-moment calculation treats m as input. The separate `MagneticSCFMACE` wrapper iterates the moments toward self-consistency; its response derivatives are not the same as those of a fixed-moment model.

In [ ]:
ensure("magnetic")

#### Architecture: condition the message passing on atomic moments


Each atomic moment `m_i` is split into magnitude and orientation features. A radial basis represents the magnitude; solid harmonics represent its direction. Those per-atom features enter the specialized interaction and product blocks at every layer, so local energy can depend on both geometry and magnetization. The output `magforces = -∂E/∂m_i` is the energy gradient with respect to the input moments.

In [ ]:
display(Image(filename=str(TUTORIAL_REPO / "tutorials/figures/architecture_magnetic.png"), width=760))

### 2.2 Theory and physical constraints

- **State variables:** atomic positions R and magnetic moments m are separate variables. Record moment units (commonly μB), magnitude bounds, and the moment convention in the data schema.
- **Spatial transformation:** under a rigid rotation, positions and moments must be transformed consistently. Magnetic moment is physically an axial vector under inversion; check the exact parity/SO(3) convention implemented by the chosen block.
- **Time reversal:** for a nonmagnetic environment without a time-reversal-breaking field, simultaneous reversal of all moments should leave a time-reversal-even energy unchanged. Spin–orbit and external-field conventions need explicit treatment.
- **Forces:** Fᵢ = −∂E/∂Rᵢ at fixed moments.
- **Magnetic forces:** fᵢᵐ = −∂E/∂mᵢ, in energy per moment unit.
- **Fixed versus relaxed moments:** if a wrapper solves m*(R), the derivative of E(R,m*(R)) includes the relaxation response. Do not label a fixed-moment derivative as the fully relaxed derivative.

### 2.3 Follow the implementation

Inspect the constructor and forward pass for the input moment representation, its radial/angular embedding, the specialized block arguments, and the derivative calculation. In this code the public training model name is MagneticScaleShiftMACE and the implementation sits in <code>mace/modules/extensions.py</code>.

In [ ]:
show_source_hits(
    MagneticScaleShiftMACE.__init__,
    needles=("super().__init__", "atomic_inter_scale", "atomic_inter_shift"),
    context=5,
)
show_source_hits(
    MagneticScaleShiftMACE.forward,
    needles=("magmom", "magforces", "get_outputs", '"energy"'),
    context=7,
    max_chars=10000,
)
read_source("mace/modules/extensions.py", 1910, 2055)

#### What the magnetic constructor changes

In the base architecture, spherical harmonics and radial functions are computed from interatomic displacement vectors. Magnetic MACE also forms angular features from each moment direction and radial features from its magnitude. These features are supplied to interaction/product modules whose signatures accept magnetic irreps. The model remains extensive through atomic-energy contributions, but the local energy can now depend on neighbouring moments.

**Source-reading task:** in the printed source, find where the moments are normalized/scaled and which constructor arguments carry magnetic irreps into the blocks. What happens to predictions when an input moment lies outside the training range? Which option controls the one-body magnetic contribution?

### 2.4 Data contract and training route

A magnetic ExtXYZ frame should carry positions, reference energy and forces as appropriate, moment vectors on every atom, and optionally magnetic-force labels. Keep all moment vectors in one declared unit and convention. Do not mix scalar total magnetization with per-site moment vectors.

**Pretraining example:** `bash tutorials/scripts/train_magnetic.sh`

- Training data: `tutorials/data/magnetic_fe2_train.extxyz`
- Validation data: `tutorials/data/magnetic_fe2_valid.extxyz`
- Defaults: CPU, 100 epochs; set `DEVICE=cuda` or `EPOCHS=...` to change them.
- The script installs only missing extension dependencies.

In [ ]:
MAGNETIC_TRAINING_OPTIONS = {
    "model": "MagneticScaleShiftMACE",
    "loss": "universal",
    "hidden_irreps": "32x0e",
    "compute_magforces": True,
    "interaction_first": "MagneticRealAgnosticSpinOrbitCoupledDensityInteractionBlock",
    "interaction": "MagneticRealAgnosticSpinOrbitCoupledDensityInteractionBlock",
    "magmom_key": "REF_magmom",
    "magforces_key": "REF_magforces",
    "magforces_weight": 5.0,
    "m_max": 4.0,                  # set from the data's supported moment range
    "num_mag_radial_basis": 8,
    "max_m_ell": 1,
}
MAGNETIC_TRAINING_OPTIONS

The bundled Fe₂ teaching set provides analytic energy, force, moment, and magnetic-force labels. Run <code>bash tutorials/scripts/train_magnetic.sh</code> for the full command. It writes the trained checkpoint to <code>tutorials/models/magnetic/</code>; the compact options below highlight the model-specific inputs.

In [ ]:
read_source("tests/extensions/magnetic/test_magmace.py", 20, 105)

> **Inspect the data:** load a magnetic ExtXYZ file and check that `REF_magmom` has shape `(N, 3)` in every frame. Plot moment magnitude against species. Check that labels cover ferromagnetic, antiferromagnetic, non-collinear and low-moment states relevant to your intended use. Compare training and test structures after symmetry-aware matching.

### 2.5 Construct and view an Fe dimer spin series

The probe holds an Fe–Fe distance fixed while changing the relative moment angle from parallel to antiparallel. This isolates one magnetic coordinate. The structure viewer draws moment vectors on both sites.

In [ ]:
def fe_dimer(distance=2.4, angle_degrees=0.0, moment=2.2, box=12.0):
    theta = np.deg2rad(angle_degrees)
    atoms = Atoms("Fe2", positions=[[5.0 - distance / 2, 6.0, 6.0],
                                    [5.0 + distance / 2, 6.0, 6.0]],
                  cell=[box, box, box], pbc=False)
    moments = np.array([[0.0, 0.0, moment],
                        [moment * np.sin(theta), 0.0, moment * np.cos(theta)]])
    atoms.new_array("magmom", moments)
    return atoms

probe = fe_dimer(distance=2.4, angle_degrees=90.0)
spin_vectors = [(probe.positions[i], probe.arrays["magmom"][i], f"m{i}", "#16805d", "μB")
                for i in range(2)]
show_structure(probe, vectors=spin_vectors, title="Fe dimer with 90° relative moment angle")

In [ ]:
distances = np.linspace(2.0, 3.5, 40)
spin_angles = [0.0, 90.0, 180.0]
fig, ax = plt.subplots(figsize=(7, 4))
for angle in spin_angles:
    ax.plot(distances, np.zeros_like(distances), label=f"relative spin angle {angle:.0f}°")
ax.set(xlabel="Fe–Fe distance (Å)", ylabel="probe coordinate (Å)",
       title="Plan a magnetic scan: vary distance and alignment independently")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

if WIDGETS_AVAILABLE and PY3DMOL_AVAILABLE:
    @widgets.interact(angle=(0.0, 180.0, 5.0))
    def explore_spin_angle(angle=90.0):
        atoms = fe_dimer(distance=2.4, angle_degrees=angle)
        vectors = [(atoms.positions[i], atoms.arrays["magmom"][i], f"m{i}", "#16805d", "μB")
                   for i in range(2)]
        return show_structure(atoms, vectors=vectors, title=f"Relative spin angle = {angle:.0f}°")

### 2.6 Checkpoint demonstration: map E(distance, relative spin angle)

Set a magnetic checkpoint trained for Fe and with a compatible cutoff, magnetic input key, and moment convention. This uses <code>MagneticMACECalculator</code>, the repository's magnetic ASE calculator. Energies below are shifted by a common reference value to make the spin-state splitting visible; do not infer absolute formation energies from this plot.

In [ ]:
mag_calc = MagneticMACECalculator(
    model_paths=str(require_model("magnetic")),
    device=DEVICE, default_dtype=MODEL_DTYPE,
    magmom_key="magmom",
)
spin_angles = [0.0, 90.0, 180.0]
distances = np.linspace(2.0, 3.5, 16)
energy_curves = {}
for angle in spin_angles:
    values = []
    for distance in distances:
        atoms = fe_dimer(distance=distance, angle_degrees=angle)
        atoms.calc = mag_calc
        values.append(atoms.get_potential_energy())
    energy_curves[angle] = np.asarray(values)
reference = energy_curves[0.0][-1]

fig, ax = plt.subplots(figsize=(7, 4.5))
for angle, energies in energy_curves.items():
    ax.plot(distances, energies - reference, marker="o", ms=3, label=f"{angle:.0f}°")
ax.set(xlabel="Fe–Fe distance (Å)", ylabel="E − E(reference) (eV)",
       title="MagneticScaleShiftMACE: fixed moments, Fe dimer")
ax.legend(title="relative moment angle")
ax.grid(alpha=0.25)
plt.show()

#### Interpret the scan carefully

The curves show what the trained model predicts for this *probe family*. A physical conclusion needs matched reference calculations, coverage of the tested moment/distance range, a consistent magnetic state definition, and uncertainty checks. If all curves overlap exactly, first check that moment arrays reach the calculator under the expected key and that the chosen checkpoint is a magnetic model.

#### Model-level rotation test

Rotate the dimer geometry and both moment vectors together. This is a useful check for a scalar-energy magnetic model. If using a spin–orbit-coupled checkpoint, rotate the crystal/environment and moments consistently; do not rotate the moments alone.

In [ ]:
atoms = fe_dimer(distance=2.4, angle_degrees=65.0)
atoms.calc = mag_calc
energy_original = atoms.get_potential_energy()
Q = Rotation.from_euler("xyz", [25.0, 40.0, 12.0], degrees=True).as_matrix()
rotated = atoms.copy()
rotated.positions = atoms.positions @ Q.T
rotated.arrays["magmom"] = atoms.arrays["magmom"] @ Q.T
rotated.calc = mag_calc
energy_rotated = rotated.get_potential_energy()
print("rotation energy residual (eV):", abs(energy_rotated - energy_original))

#### Fixed moments versus self-consistent moments

This scan holds m fixed. <code>MagneticSCFMACE</code> instead wraps a magnetic model with an iterative magnetic-moment update, caches converged moments in some workflows, and changes the derivative question. Read its forward and calculator paths before describing a relaxed magnetic force. An unrolled iteration, an implicit stationary derivative, and a fixed-input derivative are three different objects.

## Extension 4 — MACELES: augment the local energy with long-range electrostatics

### 4.1 Scientific question and architecture change

> **Architecture choice:** reuse local equivariant features to predict latent electrostatic quantities, then add an explicit long-range energy before taking forces.

A finite-cutoff local model can have difficulty representing electrostatic interactions over long distances. Latent Ewald Summation (LES) augments a short-range MLIP with an electrostatic energy constructed from learned latent atomic quantities. In the basic path, the network predicts latent charges from local features; optional configurations add dipoles, quadrupoles, induced charges and polarizabilities. An Ewald-style calculation then evaluates the long-range contribution, which is added to the local MACE energy before forces are differentiated.

Latent charges are internal variables optimized through the energy/force objective; they need not be equal to a particular quantum-chemistry partial-charge partition. The goal is to improve the energy and its derivatives, not to reproduce a non-unique charge analysis.

In [ ]:
ensure("les")

#### Architecture: add a long-range energy branch


The MACE local-energy path remains. In parallel, readouts from the final node features predict latent charges and, when configured, dipoles or higher multipoles. The LES solver combines those quantities with positions and the cell to compute a long-range energy. MACELES adds that term to the local energy before differentiating the total energy for forces. This extension adds a post-trunk branch rather than injecting features into the message-passing layers.

In [ ]:
display(Image(filename=str(TUTORIAL_REPO / "tutorials/figures/architecture_maceles.png"), width=760))

### 4.2 Theory and model boundary

Write the total energy as

$$E_\mathrm{total}(R,\mathrm{cell})=E_\mathrm{local}(R)+E_\mathrm{LES}(q,\mu,Q,\alpha,\ldots;R,\mathrm{cell}).$$

Here q, μ, Q, and α denote whichever latent quantities are enabled in the configuration. The code sums learned readout contributions, applies configured scales and constraints (for example a traceless quadrupole), calls the LES library, and adds E<sub>LES</sub> to the local energy. Forces are differentiated from the sum.

LES changes the physical scope beyond the local cutoff, but the latent quantities are still predicted from local equivariant features. It does not make every nonlocal chemical effect exact. Training data, total-charge constraints, periodic boundary conditions, self-interaction settings, and available species remain crucial.

#### Physical and practical cautions

- Keep the total charge and boundary conditions consistent with the LES configuration; inspect neutralizing-background or charge-constraint choices for periodic charged systems.
- A latent charge is not automatically an observable partial charge. Do not validate it by comparison with one arbitrary population analysis unless that is separately justified.
- For a periodic crystal, use a valid cell and the periodic LES path. For a molecule, state the non-periodic boundary convention.
- If optional multipoles or induced responses are enabled, check their equivariance, units, stability, and solver conditioning.
- MACELES requires the external <code>les</code> package. This checkout pins it in <code>requirements/les.txt</code>; the package is not part of base MACE. Install the extra only through the course/repository instructions.

### 4.3 Follow the implementation

MACELES subclasses ScaleShiftMACE, keeps the full final-layer irreps, copies compatible readout blocks for latent quantities, aggregates the per-layer predictions, calls <code>self.les(...)</code>, and adds <code>les_energy</code> before computing forces. Read these points in the source.

In [ ]:
show_source_hits(
    MACELES.__init__,
    needles=("keep_last_layer_irreps", "from les import Les", "les_readouts", "use_dipoles", "use_quads"),
    context=5,
    max_chars=10000,
)
show_source_hits(
    MACELES.forward,
    needles=("node_qs", "les_q =", "self.les(", "total_energy += les_energy", "get_outputs("),
    context=6,
    max_chars=12000,
)

#### Source questions

1. Which features are shared between the local energy readout and LES readouts?
2. Which optional outputs require 1o or 2e channels to exist in the final hidden irreps?
3. Where are predicted quadrupoles made traceless?
4. How is the cell modified for nonperiodic frames?
5. Does the forward method return total energy and LES energy separately? How does the calculator expose those values?

One subtlety: the raw model returns more intermediate outputs than the standard ASE calculator exposes. The hands-on cell below uses the raw forward output specifically to inspect latent charges and the separate LES energy.

### 4.4 Make and inspect a water dimer

The water-dimer probe varies the O···O separation while holding the monomer geometries fixed. A compatible LES checkpoint can be used to view its latent charges and energy decomposition. The plot is a model diagnostic, not a benchmark unless compared to reference data and a matched local-only model.

In [ ]:
def water_monomer_at(origin=(0.0, 0.0, 0.0)):
    origin = np.asarray(origin, dtype=float)
    local = np.array([[0.0, 0.0, 0.0], [0.96, 0.0, 0.0], [-0.24, 0.93, 0.0]])
    return origin + local


def water_dimer(separation=3.0, box=20.0):
    first = water_monomer_at((7.0, 8.0, 8.0))
    second = water_monomer_at((7.0, 8.0, 8.0 + separation))
    atoms = Atoms("OH2OH2", positions=np.vstack([first, second]),
                  cell=[box, box, box], pbc=False)
    return atoms

dimer = water_dimer(3.0)
show_structure(dimer, title="Water dimer probe at 3.0 Å O···O separation")

#### Raw model output inspection

This helper follows the same graph-building method used by the ASE calculator. It reaches into a private calculator method because the public ASE API intentionally presents total energy/forces rather than every raw network tensor. For a stable external workflow, prefer the supported calculator or evaluation CLI; use this internal path for learning how model outputs are assembled.

In [ ]:
def raw_forward_for_inspection(calc, atoms, **model_kwargs):
    model = calc.models[0]
    batch = calc._atoms_to_batch(atoms)
    with torch.no_grad():
        output = model(batch.to_dict(), training=False, compute_force=False, **model_kwargs)
    return output

les_calc = MACECalculator(model_paths=str(require_model("les")), device=DEVICE, default_dtype=MODEL_DTYPE)
les_output = raw_forward_for_inspection(les_calc, dimer)
for key in ("energy", "les_energy", "latent_charges", "latent_dipoles", "latent_quads", "latent_alphas"):
    value = les_output.get(key)
    if value is not None:
        if torch.is_tensor(value):
            print(key, tuple(value.shape), value.detach().cpu().numpy().reshape(-1)[:8])
        else:
            print(key, value)

If the checkpoint enables latent charges, colour the structure by them. Again, these are *latent electrostatic variables*. The color map is not a claim that oxygen has a unique physical partial charge or that the values should match a chosen charge partition.

In [ ]:
latent_q = les_output.get("latent_charges")
if latent_q is None:
    print("This checkpoint did not return latent_charges; inspect its les_arguments and model outputs.")
else:
    q_values = latent_q.detach().cpu().numpy().reshape(len(dimer), -1).sum(axis=1)
    show_structure(dimer, values=q_values, value_name="latent charge", value_units="e", title="MACELES latent charges on a water-dimer probe")
    print("sum of displayed latent charges:", q_values.sum())

### 4.5 Checkpoint demonstration: compare interaction curves

For each model, calculate the dimer interaction energy E<sub>AB</sub> − E<sub>A</sub> − E<sub>B</sub>. The comparison is scientifically interpretable only if the MACE and MACELES checkpoints use a matched training set, split, local architecture, and fitting protocol. The long-range model should be judged by reference interaction energies and forces over the separation range, not by the apparent smoothness of its curve.

In [ ]:
les_path = require_model("les")
short_path = require_model("short_range")
les_curve_calc = MACECalculator(model_paths=str(les_path), device=DEVICE, default_dtype=MODEL_DTYPE)
short_curve_calc = MACECalculator(model_paths=str(short_path), device=DEVICE, default_dtype=MODEL_DTYPE)


def energy_with(calc, atoms):
    atoms = atoms.copy()
    atoms.calc = calc
    return atoms.get_potential_energy()


def monomer_reference_energy(calc):
    # The isolated monomers are identical and do not depend on dimer separation.
    monomer = Atoms("OH2", positions=water_monomer_at((7.0, 8.0, 8.0)),
                    cell=[20.0] * 3, pbc=False)
    return 2.0 * energy_with(calc, monomer)


def interaction_curve(calc, separations):
    monomer_reference = monomer_reference_energy(calc)
    return np.asarray([
        energy_with(calc, water_dimer(float(separation))) - monomer_reference
        for separation in separations
    ])


separations = np.linspace(2.7, 7.0, 18)
interaction_les = interaction_curve(les_curve_calc, separations)
interaction_local = interaction_curve(short_curve_calc, separations)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(separations, 1000 * interaction_local, "o-", label="matched short-range MACE")
ax.plot(separations, 1000 * interaction_les, "s-", label="MACELES")
ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="O···O separation (Å)", ylabel="interaction energy (meV)",
       title="Water-dimer interaction: compare matched models")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


#### Reading this plot

At separations beyond the local cutoff, a strictly local model has no direct cross-fragment message; the exact behaviour depends on the model and cutoff implementation. MACELES adds electrostatic coupling, but whether it improves this curve is an empirical question. Compare with the reference at each geometry and check the force derivative. Repeat with another relative orientation to test whether the latent quantities capture the relevant electrostatics.

### 4.6 Training and validation contract

`MACELES` has a separate `les_arguments` configuration that controls which latent quantities and LES terms are active. Training may rely on energy and force supervision without reference partial-charge labels. This does **not** make the latent charges uniquely interpretable; it means the energy/force objective shapes the internal electrostatic representation.

Inspect `requirements/les.txt`, the default parser configuration and `tests/extensions/les/test_maceles.py`. Some tests depend on the optional external package; the test file documents skipped and expected-failure paths. Pin the LES revision used to train and evaluate a model.

**Matched pretraining commands:**

```bash
bash tutorials/scripts/train_maceles.sh
bash tutorials/scripts/train_maceles_short_range.sh
```

Both models use the same water-dimer structures, labels, cutoff, hidden irreps, seed and training schedule. Their data are `tutorials/data/maceles_water_dimer_train.extxyz` and `tutorials/data/maceles_water_dimer_valid.extxyz`. Defaults are CPU and 100 epochs; set `DEVICE=cuda` or `EPOCHS=...` to change them. The LES script installs its pinned optional dependency when needed.

In [ ]:
read_source("requirements/les.txt", 1, 12)
read_source("tests/extensions/les/test_maceles.py", 340, 375)

## Put the extensions side by side

| Extension | What enters or changes? | What stays shared? | Main output contract |
|---|---|---|---|
| MACEField | global field vector coupled to hidden irreps at selected stages | local MACE trunk and one scalar energy | energy, polarization, BECs, polarizability from derivatives |
| AtomicDipolesMACE | l=1 readouts of node features; optional fixed-charge baseline | MACE embedding, interactions, product basis | molecular dipole vector; direct target model |
| MagneticScaleShiftMACE | local magnetic moment inputs, magnetic embeddings and interaction/product blocks | species/geometry graph, equivariant message-passing idea, energy summation | scalar E(R,m), forces and magnetic forces |
| MACELES | extra latent-property readouts and long-range LES energy | local MACE energy path and node features | local + long-range total energy and its derivatives |

The right extension point follows the scientific contract. If a new target is only an observable vector and need not drive energy, a vector readout may be enough. If a new control variable changes the energy, it belongs in the energy path and derivatives should follow from that energy. If missing physics is long-range, an additional energy term may be preferable to forcing a finite-cutoff trunk to imitate it.

### A reusable checklist before implementing your own extension

1. **Define the quantity:** scalar, polar vector, axial vector, or higher-rank tensor? Is it an energy, a direct target, or a derivative?
2. **Write its transformation law:** rotations, inversion/parity, translations, and time reversal where relevant.
3. **Decide where it enters:** inputs, equivariant message passing, readout, total-energy assembly, loss/data, or inference interface.
4. **Choose what is held fixed:** external fields, spin moments, charge, cell, and any self-consistent variables.
5. **Map data:** labels, units, valid structures, missing-value masks, label weights, and train/test grouping.
6. **Implement the narrowest forward path:** expose only the new quantity at first; preserve the baseline model behavior.
7. **Wire the whole software route:** parser/factory, serialization, calculator, evaluation, and any accelerated or scripted path.
8. **Validate transformations and derivatives:** rotate inputs and targets, finite-difference derivatives, inspect conservation laws, and compare to matched controls.
9. **Plot a designed probe:** vary one physical degree of freedom at a time and show both structure and prediction.
10. **State limitations:** range of training data, boundary conditions, solver assumptions, and unsupported outputs.

### Final exercise — propose a fifth extension

Choose a property or physical effect relevant to your research, such as vibrational frequencies, spin–lattice coupling, chemical potential, a local tensor response, or charge-transfer energetics. In a short proposal, answer:

- What is the target, its units, and transformation law?
- Does it need to be a direct readout or a derivative of an energy?
- Which part of the architecture should change, and which trunk components can be reused?
- What data keys and structures are required?
- Which existing class in this repository is the nearest implementation pattern?
- What small scan and plot would first reveal an error?
- What matched control and reference calculation would support a scientific claim?

Then sketch the change directly onto the base architecture image, as was done in the four figures above.

## Further reading and source trail

- [Cambridge 07-MLIPs notebooks](https://gitlab.com/cam-ml/tutorials/-/tree/main/notebooks/07-mlips?ref_type=heads).
- [MACE tutorial index](https://mace-docs.readthedocs.io/en/latest/examples/tutorials.html) and [Tutorial 3: MACE Theory](https://github.com/ACEsuit/mace-tutorials/blob/main/T03_MACE_Theory.ipynb). The source schematic is MIT-licensed; see `tutorials/figures/ATTRIBUTION.md`.
- LES references: [theory](https://les.readthedocs.io/en/latest/theory.html), [MACE integration guide](https://les.readthedocs.io/en/latest/mace.html), and [source code](https://github.com/ChengUCB/les). LES is optional; pin it using `requirements/les.txt` for reproducible runs.
- Local implementation: `mace/modules/models.py`, `mace/modules/extensions.py`, `mace/modules/blocks.py`, `mace/modules/loss.py`, `mace/modules/utils.py`, `mace/tools/model_script_utils.py`, `mace/calculators/mace.py` and `tests/extensions/`.